In [1]:
import fastf1
import pandas as pd
import numpy as np
import joblib
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ── Load cached F1 session ────────────────────────────────
fastf1.Cache.enable_cache('../data/cache')
session = fastf1.get_session(2024, 'Bahrain', 'R')
session.load()

# ── Load saved models from Sets 3 + 4 ─────────────────────
tire_model = joblib.load('../models/tire_degradation_model.pkl')
compound_encoder = joblib.load('../models/compound_encoder.pkl')
driver_encoder = joblib.load('../models/driver_encoder.pkl')

print("✅ Race data loaded")
print("✅ Tire degradation model loaded")
print("✅ Encoders loaded")
print()
print(f"🏎️  Compounds available: {list(compound_encoder.classes_)}")
print(f"👥 Drivers in encoder: {len(driver_encoder.classes_)}")

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '16', '63', '4', '44', '81', '14', '18', '24', '20', '3', '22', '23', '27', '31', '10', '77', '2']


✅ Race data loaded
✅ Tire degradation model loaded
✅ Encoders loaded

🏎️  Compounds available: ['HARD', 'SOFT']
👥 Drivers in encoder: 20


In [2]:
def predict_stint(driver: str, compound: str, stint_length: int, start_lap: int = 1) -> pd.DataFrame:
    """
    Predict lap times for a full stint.
    
    Args:
        driver: e.g. 'HAM', 'VER'
        compound: 'SOFT' or 'HARD'
        stint_length: number of laps in this stint
        start_lap: race lap when stint starts
    
    Returns:
        DataFrame with predicted lap times for each lap of the stint
    """
    # Encode inputs
    driver_enc = driver_encoder.transform([driver])[0]
    compound_enc = compound_encoder.transform([compound])[0]
    
    # Build feature matrix for each lap of the stint
    stint_data = []
    for i in range(1, stint_length + 1):
        race_lap = start_lap + i - 1
        stint_data.append({
            'TyreLife': i,
            'TyreLifeSquared': i ** 2,
            'CompoundEncoded': compound_enc,
            'DriverEncoded': driver_enc,
            'CompoundAge': compound_enc * i,
            'LapNumber': race_lap
        })
    
    stint_df = pd.DataFrame(stint_data)
    
    # Predict lap times
    predicted_times = tire_model.predict(stint_df)
    
    # Build output
    result = pd.DataFrame({
        'StintLap': range(1, stint_length + 1),
        'RaceLap': range(start_lap, start_lap + stint_length),
        'TyreAge': range(1, stint_length + 1),
        'Compound': compound,
        'PredictedLapTime': predicted_times.round(3),
        'Driver': driver
    })
    
    # Calculate cumulative time + delta to fastest lap
    result['CumulativeTime'] = result['PredictedLapTime'].cumsum().round(3)
    result['DeltaToFirstLap'] = (result['PredictedLapTime'] - result['PredictedLapTime'].iloc[0]).round(3)
    
    return result

# ── Test it: Simulate Hamilton on Soft for 20 laps ────────
ham_soft_stint = predict_stint('HAM', 'SOFT', stint_length=20, start_lap=1)

print("🏎️ Hamilton — Simulated SOFT Stint (20 laps)")
print("=" * 60)
print(ham_soft_stint.to_string(index=False))
print()
print(f"📊 Stint summary:")
print(f"   Avg lap time: {ham_soft_stint['PredictedLapTime'].mean():.3f}s")
print(f"   Total stint time: {ham_soft_stint['CumulativeTime'].iloc[-1]:.3f}s")
print(f"   Degradation across stint: {ham_soft_stint['DeltaToFirstLap'].iloc[-1]:+.3f}s")

🏎️ Hamilton — Simulated SOFT Stint (20 laps)
 StintLap  RaceLap  TyreAge Compound  PredictedLapTime Driver  CumulativeTime  DeltaToFirstLap
        1        1        1     SOFT         98.155998    HAM       98.155998            0.000
        2        2        2     SOFT         98.585999    HAM      196.742004            0.430
        3        3        3     SOFT         98.456001    HAM      295.197998            0.300
        4        4        4     SOFT         98.302002    HAM      393.500000            0.146
        5        5        5     SOFT         98.401001    HAM      491.901001            0.245
        6        6        6     SOFT         98.410004    HAM      590.310974            0.254
        7        7        7     SOFT         98.723999    HAM      689.034973            0.568
        8        8        8     SOFT         98.794998    HAM      787.830017            0.639
        9        9        9     SOFT         98.668999    HAM      886.499023            0.513
     

In [3]:
# ── Compare SOFT vs HARD for a single driver ──────────────
driver = 'HAM'
stint_length = 25

soft_stint = predict_stint(driver, 'SOFT', stint_length=stint_length, start_lap=1)
hard_stint = predict_stint(driver, 'HARD', stint_length=stint_length, start_lap=1)

# Plot comparison
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f'{driver} — Lap-by-Lap Time Prediction',
        f'{driver} — Cumulative Stint Time'
    )
)

# Lap-by-lap
fig.add_trace(go.Scatter(
    x=soft_stint['StintLap'],
    y=soft_stint['PredictedLapTime'],
    name='SOFT',
    line=dict(color='#FF3333', width=3),
    mode='lines+markers'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=hard_stint['StintLap'],
    y=hard_stint['PredictedLapTime'],
    name='HARD',
    line=dict(color='#CCCCCC', width=3),
    mode='lines+markers'
), row=1, col=1)

# Cumulative time
fig.add_trace(go.Scatter(
    x=soft_stint['StintLap'],
    y=soft_stint['CumulativeTime'],
    name='SOFT (cumulative)',
    line=dict(color='#FF3333', width=3, dash='dot'),
    showlegend=False
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=hard_stint['StintLap'],
    y=hard_stint['CumulativeTime'],
    name='HARD (cumulative)',
    line=dict(color='#CCCCCC', width=3, dash='dot'),
    showlegend=False
), row=1, col=2)

fig.update_layout(
    template='plotly_dark',
    height=500,
    title_text=f'🏎️ Tire Compound Comparison — {driver} | {stint_length}-Lap Stint',
)

fig.update_xaxes(title_text='Stint Lap', row=1, col=1)
fig.update_xaxes(title_text='Stint Lap', row=1, col=2)
fig.update_yaxes(title_text='Predicted Lap Time (s)', row=1, col=1)
fig.update_yaxes(title_text='Cumulative Time (s)', row=1, col=2)

fig.show()

# ── Summary stats ─────────────────────────────────────────
print(f"\n📊 STINT COMPARISON ({stint_length} laps)")
print("=" * 60)
print(f"SOFT total: {soft_stint['CumulativeTime'].iloc[-1]:.3f}s | avg: {soft_stint['PredictedLapTime'].mean():.3f}s")
print(f"HARD total: {hard_stint['CumulativeTime'].iloc[-1]:.3f}s | avg: {hard_stint['PredictedLapTime'].mean():.3f}s")

delta = soft_stint['CumulativeTime'].iloc[-1] - hard_stint['CumulativeTime'].iloc[-1]
faster = "SOFT" if delta < 0 else "HARD"
print(f"\n🏆 Faster compound: {faster} by {abs(delta):.3f}s over {stint_length} laps")


📊 STINT COMPARISON (25 laps)
SOFT total: 2453.294s | avg: 98.132s
HARD total: 2437.593s | avg: 97.504s

🏆 Faster compound: HARD by 15.701s over 25 laps


In [4]:
PIT_STOP_TIME_LOSS = 22.0   # seconds lost during pit stop (industry standard)
RACE_LENGTH = 57            # 2024 Bahrain GP race distance

def simulate_1stop_strategy(
    driver: str,
    pit_lap: int,
    stint1_compound: str,
    stint2_compound: str,
    race_length: int = RACE_LENGTH
) -> dict:
    """
    Simulate a 1-stop race strategy.
    Returns total race time including pit stop loss.
    """
    # Stint 1: from race lap 1 to pit_lap
    stint1_len = pit_lap
    stint1 = predict_stint(driver, stint1_compound, stint_length=stint1_len, start_lap=1)
    
    # Stint 2: from pit_lap + 1 to end of race
    stint2_len = race_length - pit_lap
    stint2 = predict_stint(driver, stint2_compound, stint_length=stint2_len, start_lap=pit_lap + 1)
    
    total_time = (
        stint1['CumulativeTime'].iloc[-1]
        + stint2['CumulativeTime'].iloc[-1]
        + PIT_STOP_TIME_LOSS
    )
    
    return {
        'PitLap': pit_lap,
        'Stint1Compound': stint1_compound,
        'Stint2Compound': stint2_compound,
        'Stint1Laps': stint1_len,
        'Stint2Laps': stint2_len,
        'Stint1Time': round(stint1['CumulativeTime'].iloc[-1], 3),
        'Stint2Time': round(stint2['CumulativeTime'].iloc[-1], 3),
        'PitLoss': PIT_STOP_TIME_LOSS,
        'TotalRaceTime': round(total_time, 3)
    }

# ── Find optimal pit lap for SOFT→HARD strategy ──────────
driver = 'HAM'
strategies = []

# Test pit laps from 10 to 45 (realistic pit windows)
for pit_lap in range(10, 46):
    result = simulate_1stop_strategy(
        driver=driver,
        pit_lap=pit_lap,
        stint1_compound='SOFT',
        stint2_compound='HARD'
    )
    strategies.append(result)

strategies_df = pd.DataFrame(strategies)
optimal = strategies_df.loc[strategies_df['TotalRaceTime'].idxmin()]

print(f"🏎️  PIT WINDOW ANALYSIS — {driver}")
print(f"   Strategy: SOFT → HARD (1-stop)")
print(f"   Race length: {RACE_LENGTH} laps")
print("=" * 60)
print(f"\n🏆 OPTIMAL PIT LAP: Lap {int(optimal['PitLap'])}")
print(f"   Stint 1: {int(optimal['Stint1Laps'])} laps on SOFT ({optimal['Stint1Time']}s)")
print(f"   Pit stop: {optimal['PitLoss']}s lost")
print(f"   Stint 2: {int(optimal['Stint2Laps'])} laps on HARD ({optimal['Stint2Time']}s)")
print(f"   ⏱️  Total race time: {optimal['TotalRaceTime']}s")
print()
print("📊 Top 5 strategies:")
print(strategies_df.nsmallest(5, 'TotalRaceTime')[['PitLap', 'Stint1Time', 'Stint2Time', 'TotalRaceTime']].to_string(index=False))
print()
print("📉 Worst 3 strategies (avoid these):")
print(strategies_df.nlargest(3, 'TotalRaceTime')[['PitLap', 'Stint1Time', 'Stint2Time', 'TotalRaceTime']].to_string(index=False))

🏎️  PIT WINDOW ANALYSIS — HAM
   Strategy: SOFT → HARD (1-stop)
   Race length: 57 laps

🏆 OPTIMAL PIT LAP: Lap 34
   Stint 1: 34 laps on SOFT (3331.861083984375s)
   Pit stop: 22.0s lost
   Stint 2: 23 laps on HARD (2193.785888671875s)
   ⏱️  Total race time: 5547.64697265625s

📊 Top 5 strategies:
 PitLap  Stint1Time  Stint2Time  TotalRaceTime
     34 3331.861084 2193.785889    5547.646973
     35 3429.397949 2096.284912    5547.682129
     33 3234.367920 2292.127930    5548.496094
     36 3526.851074 1999.720947    5548.571777
     37 3624.197021 1902.437988    5548.634766

📉 Worst 3 strategies (avoid these):
 PitLap  Stint1Time  Stint2Time  TotalRaceTime
     10  984.789001 4562.358887    5569.147949
     11 1083.055054 4462.104980    5567.160156
     15 1475.735962 4068.806885    5566.542969


In [5]:
fig = go.Figure()

# Plot strategy curve
fig.add_trace(go.Scatter(
    x=strategies_df['PitLap'],
    y=strategies_df['TotalRaceTime'],
    mode='lines+markers',
    name='Total Race Time',
    line=dict(color='#00d2ff', width=3),
    marker=dict(size=8)
))

# Highlight optimal pit lap
fig.add_trace(go.Scatter(
    x=[optimal['PitLap']],
    y=[optimal['TotalRaceTime']],
    mode='markers',
    name='🏆 Optimal',
    marker=dict(color='#00ff88', size=20, symbol='star',
                line=dict(width=2, color='white'))
))

# Add strategy window shading (top 5 strategies)
top5 = strategies_df.nsmallest(5, 'TotalRaceTime')
fig.add_vrect(
    x0=top5['PitLap'].min() - 0.5,
    x1=top5['PitLap'].max() + 0.5,
    fillcolor='#00ff88',
    opacity=0.1,
    line_width=0,
    annotation_text="Strategy Window",
    annotation_position="top left"
)

fig.update_layout(
    title=f'🏎️ Pit Window Analysis — {driver} | SOFT → HARD Strategy<br>'
          f'<sub>Optimal: Lap {int(optimal["PitLap"])} | Race Length: {RACE_LENGTH} laps</sub>',
    xaxis_title='Pit Lap',
    yaxis_title='Total Race Time (seconds)',
    template='plotly_dark',
    height=500,
    hovermode='x unified'
)

fig.show()

print(f"\n💡 STRATEGY INSIGHT:")
print(f"   Window: Pit between Lap {int(top5['PitLap'].min())} and Lap {int(top5['PitLap'].max())}")
print(f"   Risk: Pitting before Lap {int(top5['PitLap'].min())} costs >1s")
print(f"   Risk: Pitting after Lap {int(top5['PitLap'].max())} costs >1s")


💡 STRATEGY INSIGHT:
   Window: Pit between Lap 33 and Lap 37
   Risk: Pitting before Lap 33 costs >1s
   Risk: Pitting after Lap 37 costs >1s


In [6]:
import os
os.makedirs('../data/strategy', exist_ok=True)

# Save strategy analysis
strategies_df.to_csv('../data/strategy/ham_pit_window_analysis.csv', index=False)

# Save optimal strategy summary
strategy_summary = {
    'driver': driver,
    'race': '2024 Bahrain GP',
    'race_length_laps': RACE_LENGTH,
    'strategy_type': 'SOFT → HARD (1-stop)',
    'optimal_pit_lap': int(optimal['PitLap']),
    'stint1_compound': 'SOFT',
    'stint1_laps': int(optimal['Stint1Laps']),
    'stint1_time_seconds': float(optimal['Stint1Time']),
    'pit_stop_loss_seconds': PIT_STOP_TIME_LOSS,
    'stint2_compound': 'HARD',
    'stint2_laps': int(optimal['Stint2Laps']),
    'stint2_time_seconds': float(optimal['Stint2Time']),
    'total_race_time_seconds': float(optimal['TotalRaceTime']),
    'pit_window_start': int(top5['PitLap'].min()),
    'pit_window_end': int(top5['PitLap'].max()),
}

import json
with open('../data/strategy/ham_optimal_strategy.json', 'w') as f:
    json.dump(strategy_summary, f, indent=2)

print("✅ Strategy analysis saved")
print(f"📁 Full analysis: ../data/strategy/ham_pit_window_analysis.csv")
print(f"📁 Optimal strategy: ../data/strategy/ham_optimal_strategy.json")
print()
print("📋 OPTIMAL STRATEGY SUMMARY")
print(json.dumps(strategy_summary, indent=2))

✅ Strategy analysis saved
📁 Full analysis: ../data/strategy/ham_pit_window_analysis.csv
📁 Optimal strategy: ../data/strategy/ham_optimal_strategy.json

📋 OPTIMAL STRATEGY SUMMARY
{
  "driver": "HAM",
  "race": "2024 Bahrain GP",
  "race_length_laps": 57,
  "strategy_type": "SOFT \u2192 HARD (1-stop)",
  "optimal_pit_lap": 34,
  "stint1_compound": "SOFT",
  "stint1_laps": 34,
  "stint1_time_seconds": 3331.861083984375,
  "pit_stop_loss_seconds": 22.0,
  "stint2_compound": "HARD",
  "stint2_laps": 23,
  "stint2_time_seconds": 2193.785888671875,
  "total_race_time_seconds": 5547.64697265625,
  "pit_window_start": 33,
  "pit_window_end": 37
}
